In [38]:
# Importing 
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
# Get the current directory where your script is located
base_path = os.getcwd()

# Join the path to the dataset folder
train_path = os.path.join(base_path, 'dataset', 'train.csv')

# Load the data
train_df = pd.read_csv(train_path)
print("Data loaded successfully!")


Data loaded successfully!


In [25]:
Null_df = train_df.isnull().sum()
print("\nNull Values in each columns : ")
print(Null_df)
print("\nShape of the data")
print(train_df.shape)


Null Values in each columns : 
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

Shape of the data
(614, 13)


In [ ]:
# Categorical Col Filled with mode
Column_names = ["Gender","Married","Dependents","Self_Employed","Credit_History"]
for col in Column_names:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

# Numerical Col Filled with median
col_me = ["LoanAmount","Loan_Amount_Term"]
for coll in col_me :
    train_df[coll] = train_df[coll].fillna(train_df[coll].median())
train_df = train_df.drop("Loan_ID",axis = 1) # dropping Loan ID its not necessary for predicting loan

In [27]:
train_df.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [28]:
# Split the data in feature and target
X = train_df.drop("Loan_Status", axis=1) #FEATURE DATA
y = train_df["Loan_Status"]#TARGET DATA

# splited the data in Training and Test
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, stratify=y,random_state=42)


In [29]:
# encoding  for X_train
ct = ColumnTransformer(transformers=[("encoder",OneHotEncoder(handle_unknown="ignore", sparse_output=False),["Gender", "Married","Education","Dependents","Self_Employed", "Property_Area"])],remainder="passthrough")
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

#label encoding for target
leb = LabelEncoder()
y_train = leb.fit_transform(y_train)
y_test = leb.transform(y_test)



In [30]:
# To see the new column order after encoding
feature_names = ct.get_feature_names_out()
feature_names = [names.split("__")[1]for names in feature_names]
X_train = pd.DataFrame(X_train,columns=feature_names)
X_test = pd.DataFrame(X_test, columns=feature_names)

In [31]:
# finding the best depth for model
for depth in [1, 2, 3, 4, 5, 6, 7, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train = model.score(X_train, y_train)
    test  = model.score(X_test, y_test)
    print(f"depth={str(depth):4} → train={train:.4f}  test={test:.4f}")

depth=1    → train=0.7984  test=0.8537
depth=2    → train=0.8045  test=0.8537
depth=3    → train=0.8086  test=0.8455
depth=4    → train=0.8086  test=0.8455
depth=5    → train=0.8248  test=0.8211
depth=6    → train=0.8432  test=0.8130
depth=7    → train=0.8656  test=0.7967
depth=None → train=1.0000  test=0.7398


In [40]:
model =  DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state = 42)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

In [ ]:
print("\nConfusion Matrix: ")
print(confusion_matrix(y_test,y_pred))
print("--"*40)
print("\nClassification Report : ")
print(f"{classification_report(y_test,y_pred)}")
# Model is Biased toward approving Loans


Confusion Matrix: 
[[22 16]
 [ 8 77]]
--------------------------------------------------------------------------------

Classification Report : 
              precision    recall  f1-score   support

           0       0.73      0.58      0.65        38
           1       0.83      0.91      0.87        85

    accuracy                           0.80       123
   macro avg       0.78      0.74      0.76       123
weighted avg       0.80      0.80      0.80       123



In [37]:
# Model choose Credit History as Root node
importance = model.feature_importances_
feature_names = X_train.columns

ft_imp = pd.Series(importance, index=feature_names).sort_values(ascending=False)
print(ft_imp.head(10))

Credit_History            0.633474
CoapplicantIncome         0.235683
LoanAmount                0.083364
ApplicantIncome           0.047480
Married_Yes               0.000000
Married_No                0.000000
Gender_Male               0.000000
Gender_Female             0.000000
Education_Graduate        0.000000
Education_Not Graduate    0.000000
dtype: float64
